In [1]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np

class VirtualSensor:
    def __init__(self):
        model_path = 'pose_landmarker_lite.task'
        base_options = python.BaseOptions(model_asset_path=model_path)
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            output_segmentation_masks=False
        )
        self.detector = vision.PoseLandmarker.create_from_options(options)

    def process_frame(self, frame):
        # Convert BGR to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # Create MP Image
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        # Detect
        results = self.detector.detect(mp_image)
        
        # Extract 33 landmarks (x, y, z)
        if results.pose_landmarks:
            landmarks = []
            for lm in results.pose_landmarks[0]:  # Assuming one person
                landmarks.append([lm.x, lm.y, lm.z])
            return np.array(landmarks)
        return None
def mediapipe_to_standard(results):
    if not results.pose_world_landmarks:
        return None
        
    lms = results.pose_world_landmarks.landmark
    # Initialize our 15x3 array
    standard = np.zeros((15, 3))
    
    # 1. Direct Mappings (Head, Shoulders, Elbows, Wrists, Hips, Knees, Ankles)
    direct_map = {0:0, 1:11, 2:12, 3:13, 4:14, 5:15, 6:16, 
                  7:23, 8:24, 9:25, 10:26, 11:27, 12:28}
    
    for model_idx, mp_idx in direct_map.items():
        standard[model_idx] = [lms[mp_idx].x, lms[mp_idx].y, lms[mp_idx].z]
        
    # 2. Calculated Mappings (The "Virtual" Spine Joints)
    # Spine Base = Midpoint between Hips
    standard[13] = (standard[7] + standard[8]) / 2
    
    # Spine Mid = Center of the Torso (Avg of shoulders and hips)
    standard[14] = (standard[1] + standard[2] + standard[7] + standard[8]) / 4
    
    return standard # This is now ready for your GCN
# Usage Example (Data Collection Loop)
def collect_data(video_path, save_path):
    cap = cv2.VideoCapture(video_path)
    sensor = VirtualSensor()
    skeleton_sequence = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        data = sensor.process_frame(frame)
        if data is not None:
            skeleton_sequence.append(data)
    
    # Save as numpy array: Shape (Frames, 33, 3)
    np.save(save_path, np.array(skeleton_sequence))
    cap.release()
    print(f"Data saved to {save_path}")

# Run this for your rehab videos
# collect_data('patient_exercise.mp4', 'patient_data.npy')

In [2]:
import numpy as np

class Graph:
    def __init__(self, strategy='spatial'):
        self.num_node = 33
        self.edges = self.get_mediapipe_edges()
        
        # Determine the Strategy
        if strategy == 'uniform':
            self.A = self.get_uniform_graph()
        elif strategy == 'spatial':
            self.A = self.get_spatial_graph()
        else:
            raise ValueError("Strategy must be 'uniform' or 'spatial'")

    def get_mediapipe_edges(self):
        # MediaPipe connections
        pairs = [
            (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8),
            (9, 10), (11, 12), (11, 13), (13, 15), (15, 17), (15, 19), (15, 21),
            (12, 14), (14, 16), (16, 18), (16, 20), (16, 22),
            (11, 23), (12, 24), (23, 24),
            (23, 25), (25, 27), (27, 29), (27, 31), (29, 31),
            (24, 26), (26, 28), (28, 30), (28, 32), (30, 32)
        ]
        return pairs

    def get_uniform_graph(self):
        # 1. Initialize
        A = np.zeros((self.num_node, self.num_node))
        
        # 2. Add edges and self-loops
        for i, j in self.edges:
            A[i, j] = 1
            A[j, i] = 1
        for i in range(self.num_node):
            A[i, i] = 1
            
        # 3. Normalize
        A_norm = self.normalize_adjacency(A)
        
        # 4. Stack (Duplicates the same matrix)
        return np.stack([A_norm, A_norm, A_norm], axis=0)

    def get_spatial_graph(self):
        # --- ST-GCN "Spatial" Strategy ---
        
        # 1. Define the "Center" of the body
        # For MediaPipe, Node 0 (Nose) or Node 23/24 (Hips) are good candidates.
        # We use Node 0 as the root for hop-distance calculation.
        center_node = 0 
        
        # 2. Calculate Hop Distances from Center
        hop_dis = self.get_hop_distance(self.num_node, self.edges, center_node)
        
        # 3. Initialize 3 Matrices: [Root, Centripetal, Centrifugal]
        # A_root: The node itself
        # A_close: Neighbor is CLOSER to center than root
        # A_far: Neighbor is FARTHER from center than root
        A = np.zeros((3, self.num_node, self.num_node))
        
        # 4. Fill the Matrices
        # a) Self-loops (Root)
        for i in range(self.num_node):
            A[0, i, i] = 1
            
        # b) Neighbors
        for i, j in self.edges:
            # Check edge (i -> j)
            self._fill_spatial_edge(A, i, j, hop_dis)
            # Check edge (j -> i)
            self._fill_spatial_edge(A, j, i, hop_dis)
            
        # 5. Normalize each matrix separately
        A_spatial = []
        for i in range(3):
            A_spatial.append(self.normalize_adjacency(A[i]))
            
        return np.stack(A_spatial, axis=0)

    def _fill_spatial_edge(self, A, node_root, node_neighbor, hop_dis):
        # Determine which subset the neighbor belongs to relative to root
        if hop_dis[node_neighbor] < hop_dis[node_root]:
            # Neighbor is closer to center (Centripetal) -> Subset 1
            A[1, node_root, node_neighbor] = 1
        elif hop_dis[node_neighbor] > hop_dis[node_root]:
            # Neighbor is farther from center (Centrifugal) -> Subset 2
            A[2, node_root, node_neighbor] = 1
        else:
            # Same distance (rare in trees, but possible) -> Treat as Root/Subset 0
            A[0, node_root, node_neighbor] = 1

    def get_hop_distance(self, num_node, edges, center_node):
        # Standard BFS to find shortest path from center to all nodes
        adj = [[] for _ in range(num_node)]
        for i, j in edges:
            adj[i].append(j)
            adj[j].append(i)
            
        dist = [-1] * num_node
        dist[center_node] = 0
        queue = [center_node]
        
        while queue:
            curr = queue.pop(0)
            for neighbor in adj[curr]:
                if dist[neighbor] == -1:
                    dist[neighbor] = dist[curr] + 1
                    queue.append(neighbor)
        return dist

    def normalize_adjacency(self, A):
        # Prevent division by zero
        Dl = np.sum(A, 0)
        num_node = A.shape[0]
        Dn = np.zeros((num_node, num_node))
        for i in range(num_node):
            if Dl[i] > 0:
                Dn[i, i] = Dl[i]**(-1)
        AD = np.dot(A, Dn)
        return AD

import numpy as np

class Graph:
    def __init__(self, strategy='uniform'):
        self.edges = self.get_mediapipe_edges()
        self.num_node = 33
        self.self_loops = [(i, i) for i in range(self.num_node)]
        A_single = self.get_adjacency_matrix(strategy)  # (33, 33)
        self.A = np.stack([A_single, A_single, A_single], axis=0)  # (3, 33, 33)

    def get_mediapipe_edges(self):
        # MediaPipe Pose Connectivity (Simplified subset)
        # 0:Nose, 11:L_Shoulder, 12:R_Shoulder, 23:L_Hip, 24:R_Hip, etc.
        # See MediaPipe documentation for full map
        pairs = [
            (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8), # Face
            (9, 10), (11, 12), (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), # Left Arm
            (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), # Right Arm
            (11, 23), (12, 24), (23, 24), # Torso
            (23, 25), (25, 27), (27, 29), (27, 31), (29, 31), # Left Leg
            (24, 26), (26, 28), (28, 30), (28, 32), (30, 32)  # Right Leg
        ]
        return pairs

    def get_adjacency_matrix(self, strategy):
        # Create standard Adjacency Matrix
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges:
            A[i, j] = 1
            A[j, i] = 1
        return self.normalize_adjacency(A)

    def normalize_adjacency(self, A):
        # Simple normalization: D^-1 * A
        Dl = np.sum(A, 0)
        num_node = A.shape[0]
        Dn = np.zeros((num_node, num_node))
        for i in range(num_node):
            if Dl[i] > 0:
                Dn[i, i] = Dl[i]**(-1)
        AD = np.dot(A, Dn)
        return AD

In [3]:
pip install mediapipe

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip show mediapipe

Name: mediapipe
Version: 0.10.31
Summary: MediaPipe is the simplest way for researchers and developers to build world-class ML solutions and applications for mobile, edge, cloud and the web.
Home-page: https://github.com/google/mediapipe
Author: The MediaPipe Authors
Author-email: mediapipe@google.com
License: Apache 2.0
Location: c:\Users\theda\anaconda3\envs\HAR\Lib\site-packages
Requires: absl-py, flatbuffers, numpy, sounddevice
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
import numpy as np
import json
import os
import glob
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

class BlazePoseBodyGraph():
    def __init__(self):
        # We use joints 11 through 32 (Total 22 joints)
        # 0 in our graph = 11 in MediaPipe (Left Shoulder)
        self.num_node = 33
        self.edges = [
            (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8), # Face
            (9, 10), (11, 12), (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), # Left Arm
            (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), # Right Arm
            (11, 23), (12, 24), (23, 24), # Torso
            (23, 25), (25, 27), (27, 29), (27, 31), (29, 31), # Left Leg
            (24, 26), (26, 28), (28, 30), (28, 32), (30, 32)  # Right Leg
        ]
        self.A = self.get_adjacency_matrix()

    def get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges:
            A[i, j] = A[j, i] = 1
        A = A + np.eye(self.num_node)
        D = np.diag(np.power(A.sum(1), -0.5))
        return torch.from_numpy(D @ A @ D).float()

In [6]:
import os
import glob
import json
import torch
import numpy as np
import xml.etree.ElementTree as ET
from torch.utils.data import Dataset

# --- ANVIL PARSER ---
def get_label_from_anvil(anvil_path):
    try:
        tree = ET.parse(anvil_path)
        root = tree.getroot()
        for track in root.findall(".//track[@name='Global evaluation']"):
            for el in track.findall("el"):
                attribute = el.find("attribute[@name='evaluation']")
                if attribute is not None:
                    status = attribute.text.strip().lower()
                    return 0 if status == "correct" else 1
        return 1
    except Exception:
        return 1

# --- DATASET CLASS ---
class KeraalDataset(Dataset):
    def __init__(self, root_dir, target_len=60):
        self.root_dir = root_dir
        self.target_len = target_len
        self.label_map = {"RTK": 0, "ELK": 1, "CTK": 2}
        
        # 1. Update Glob to find JSONs in ALL group folders
        # Matches: group1A, group1B, group2A, group2B, group3
        self.json_files = glob.glob(os.path.join(root_dir, "group*", "blazepose", "*.json"))
        
        # Exact spelling matches your JSON
        self.joint_names = [
            "Nose", "Left_eye_inner", "Left_eye", "Left_eye_outer",
            "Right_eye_inner", "Right_eye", "Right_eye_outer",
            "Left_ear", "Right_ear", "Mouth_left", "Mouth_right",
            "Left_shoulder", "Right_shoulder", "Left_elbow", "Right_elbow",
            "Left_wrist", "Right_wrist", "Left_pinky", "Right_pinky",
            "Left_index", "Right_index", "Left_thumb", "Right_thumb",
            "Left_hip", "Right_hip", "Left_knee", "Right_knee",
            "Left_ankle", "Right_ankle", "Left_heel", "Right_heel",
            "Left_foot_index", "Right_foot_index"
        ]

    def __len__(self):
        return len(self.json_files)

    def __getitem__(self, idx):
        json_path = self.json_files[idx]
        filename = os.path.basename(json_path)
        
        # Identify parent directory (e.g., "group1A", "group3")
        # Structure: root / groupX / blazepose / file.json
        group_name = os.path.basename(os.path.dirname(os.path.dirname(json_path)))

        # --- 1. DETERMINE LABELS & EXERCISE ---
        correctness = 1 # Default to incorrect
        exercise_name = ""

        # Logic for Group 3 (Different naming convention & Always Error)
        if group_name == "group3":
            # G3 filename: G3_ExerciseName_ParticipantId_...
            parts = filename.split('-')
            exercise_name = parts[2] # e.g. "RTK"
            correctness = 1 # Group 3 simulates errors
            
        # Logic for Groups 1A, 1B, 2A, 2B
        else:
            # Standard filename: GroupId-Modality-ExerciseName-SubjectId-RecordingId
            parts = filename.replace(".json", "").split('-')
            
            # Safety check for malformed filenames
            if len(parts) >= 3:
                exercise_name = parts[2]
            
            # UNLABELED GROUPS (1B, 2B) -> Assume Correct (0)
            if "B" in group_name: 
                correctness = 0
            
            # LABELED GROUPS (1A, 2A) -> Check ANVIL
            else:
                rec_id = parts[-1] 
                num_id = ''.join(filter(str.isdigit, rec_id))
                
                # Look for annotations in the specific group folder
                # We search recursively in the group folder for the .anvil file
                # because the readme says 'annotations' but your code used 'annotatorA'
                search_path = os.path.join(self.root_dir, group_name, "**", f"*{num_id}*.anvil")
                found_anvil = glob.glob(search_path, recursive=True)
                
                if found_anvil:
                    correctness = get_label_from_anvil(found_anvil[0])
                else:
                    # Fallback if annotation missing (Default to Incorrect/1 or Correct/0 based on preference)
                    correctness = 1 

        # Map Exercise Name to ID
        exercise_id = self.label_map.get(exercise_name, 0)
        
        # Compute Final Label (0-5)
        label = exercise_id * 2 + correctness

        # --- 2. LOAD JSON DATA ---
        with open(json_path, 'r') as f:
            raw_data = json.load(f)
        
        data_dict = raw_data.get("positions", raw_data)

        # --- 3. SORT FRAMES ---
        frames_map = []
        for k in data_dict.keys():
            try:
                frames_map.append((float(k), k))
            except ValueError:
                continue
        
        frames_map.sort(key=lambda x: x[0])

        # --- 4. EXTRACT SKELETONS ---
        sequence = []
        for _, original_key in frames_map:
            current_frame = data_dict[original_key]
            frame_joints = []
            for j_name in self.joint_names:
                coords = current_frame.get(j_name, [0.0, 0.0, 0.0])
                frame_joints.append(coords)
            sequence.append(frame_joints)
            
        sequence = np.array(sequence)

        # --- 5. NORMALIZATION ---
        if len(sequence) < 1:
            return torch.zeros(3, self.target_len, 33), label

        # Temporal Interpolation
        T = sequence.shape[0]
        final_seq = np.zeros((self.target_len, 33, 3))
        curr_x = np.linspace(0, T, T)
        target_x = np.linspace(0, T, self.target_len)
        
        for j in range(33):
            for c in range(3):
                final_seq[:, j, c] = np.interp(target_x, curr_x, sequence[:, j, c])

        # Hip Center Normalization
        hip_center = (final_seq[:, 23, :] + final_seq[:, 24, :]) / 2
        final_seq = final_seq - hip_center[:, np.newaxis, :]

        # Transpose
        final_seq = np.transpose(final_seq, (2, 0, 1))
        
        return torch.FloatTensor(final_seq), label

In [7]:
from torch.utils.data import DataLoader, random_split

# Initialize the full dataset
full_dataset = KeraalDataset(root_dir="C:\\Users\\theda\\downloads\\HAR_REHAB\\Dataset")

# Split: 80% for training, 20% for validation
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Create separate DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

class RehabRecognizer(nn.Module):
    def __init__(self, num_classes, graph):
        super().__init__()
        self.graph = graph
        A = self.graph.A
        self.l1 = STGCNBlock(3, 64, A)
        self.l2 = STGCNBlock(64, 128, A, stride=2)
        self.l3 = STGCNBlock(128, 256, A, stride=2)
        self.fc = nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.l1(x)
        x = self.l2(x)
        x = self.l3(x)
        x = F.avg_pool2d(x, x.size()[2:]).view(x.size(0), -1)
        return self.fc(x)

class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, A, stride=1):
        super().__init__()
        self.register_buffer('A', A)
        self.gcn = nn.Conv2d(in_channels, out_channels, 1)
        self.tcn = nn.Sequential(nn.BatchNorm2d(out_channels), nn.ReLU(),
                                 nn.Conv2d(out_channels, out_channels, (9,1), (stride,1), (4,0)),
                                 nn.BatchNorm2d(out_channels))
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.tcn(self.gcn(x))

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
#from graph import Graph

class GraphConvolution(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, A):
        super().__init__()
        self.kernel_size = kernel_size
        self.conv = nn.Conv2d(in_channels, out_channels * kernel_size, kernel_size=(1, 1), padding=0)
        self.register_buffer('A', torch.from_numpy(A).float())

    def forward(self, x):
        # x shape: (Batch, Channels, Time, Nodes)
        x = self.conv(x)
        n, kc, t, v = x.size()
        x = x.view(n, self.kernel_size, kc // self.kernel_size, t, v)
        # Apply graph convolution (einsum is faster for matrix multiplication here)
        # Equivalent to: sum(A * x)
        x = torch.einsum('nkctv,kvw->nctw', (x, self.A))
        return x.contiguous()

class ST_GCN_Block(nn.Module):
    def __init__(self, in_channels, out_channels, A, stride=1, residual=True):
        super().__init__()
        self.gcn = GraphConvolution(in_channels, out_channels, 3, A)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, (9, 1), (stride, 1), padding=(4, 0)),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(0.1, inplace=True),
        )
        
        if not residual:
            self.residual = lambda x: 0
        elif (in_channels == out_channels) and (stride == 1):
            self.residual = lambda x: x
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        res = self.residual(x)
        x = self.gcn(x)
        x = self.tcn(x)
        return self.relu(x + res)

class RehabActionRecognizer(nn.Module):
    def __init__(self, num_classes, in_channels=3):
        super().__init__()
        self.graph = Graph()
        self.A = self.graph.A
        
        # Network Architecture
        self.data_bn = nn.BatchNorm1d(in_channels * 33)
        self.layers = nn.ModuleList([
            ST_GCN_Block(in_channels, 64, self.A, residual=False),
            ST_GCN_Block(64, 64, self.A),
            ST_GCN_Block(64, 64, self.A),
            ST_GCN_Block(64, 128, self.A, stride=2),
            ST_GCN_Block(128, 128, self.A),
            ST_GCN_Block(128, 128, self.A),
            ST_GCN_Block(128, 256, self.A, stride=2),
            ST_GCN_Block(256, 256, self.A),
            ST_GCN_Block(256, 256, self.A),
        ])
        
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        # Input shape: (N, C, T, V) -> (Batch, 3, Frames, 33)
        N, C, T, V = x.size()
        x = x.permute(0, 3, 1, 2).contiguous().view(N, V * C, T)
        x = self.data_bn(x)
        x = x.view(N, V, C, T).permute(0, 2, 3, 1).contiguous()
        
        for layer in self.layers:
            x = layer(x)
            
        # Global Pooling
        x = F.avg_pool2d(x, x.size()[2:])
        x = x.view(N, -1)
        x = self.fc(x)
        return x

In [9]:
def train_blazepose():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Setup Data
    dataset = KeraalDataset(root_dir="C:\\Users\\theda\\downloads\\HAR_REHAB\\Dataset") # Point this to where group1A folder is
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True)
    
    # 2. Setup Model
    graph = BlazePoseBodyGraph()
    # Define STGCN block (same as previous examples)
    model = RehabActionRecognizer(num_classes=6).to(device) # 6 classes: 3 exercises x 2 correctness
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    
    best_avg_acc = 0.0
    model_path = "blazepose_rehab_100epoch.pth"
    
    # 3. Train
    for epoch in range(1, 100):
    # --- TRAINING PHASE ---
        model.train()
        train_loss, train_correct = 0, 0
        for data, labels in train_loader:
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
        
            train_loss += loss.item()
            train_correct += (output.argmax(1) == labels).sum().item()

    # --- VALIDATION PHASE ---
        model.eval()
        val_loss, val_correct = 0, 0
        with torch.no_grad(): # No gradient calculation for validation
            for data, labels in val_loader:
                output = model(data)
                loss = criterion(output, labels)
                val_loss += loss.item()
                val_correct += (output.argmax(1) == labels).sum().item()
        
        current_avg_acc = 100 * val_correct / val_size +100* train_correct/train_size
        if current_avg_acc > best_avg_acc:
            print(f"Average accuracy improved ({best_avg_acc:.2f}% --> {current_avg_acc/2:.2f}%). Saving model...")
            best_avg_acc = current_avg_acc
            torch.save(model.state_dict(), model_path)
        else:
            print(f"Average accuracy did not improve from {best_avg_acc:.2f}%")
    # Print comparative metrics
        print(f"Epoch {epoch}")
        print(f"Train Acc: {100 * train_correct / train_size:.2f}% | Train Loss: {train_loss/len(train_loader):.4f}")
        print(f"Val Acc:   {100 * val_correct / val_size:.2f}% | Val Loss:   {val_loss/len(val_loader):.4f}")
        print("-" * 30)
    
    

# Helper class for the model (Simplified)


 train_blazepose()

In [10]:
data, labels = next(iter(train_loader))

In [11]:
labels

tensor([4, 0, 1, 4, 1, 0, 2, 3])

In [12]:
all_labels = []
for _, labels in train_loader:
    all_labels.extend(labels.tolist())
print("Train label distribution:", torch.bincount(torch.tensor(all_labels)))

all_val_labels = []
for _, labels in val_loader:
    all_val_labels.extend(labels.tolist())
print("Val label distribution:", torch.bincount(torch.tensor(all_val_labels)))

Train label distribution: tensor([533, 178, 461, 183, 561, 180])
Val label distribution: tensor([124,  53, 110,  50, 141,  47])


In [13]:
# Config
WINDOW_SIZE = 50  # Number of frames model expects (T)
NUM_CLASSES = 6   # 3 exercises x 2 correctness
CLASSES = ["RTK Correct", "RTK Incorrect", "ELK Correct", "ELK Incorrect", "CTK Correct", "CTK Incorrect"]

In [14]:
import torch
import cv2
import numpy as np


import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# 1. Path to the file you just downloaded
model_path = 'pose_landmarker_lite.task'

# 2. Create options
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=True
)

# 3. Create the detector
detector = vision.PoseLandmarker.create_from_options(options)

print("Lite model loaded successfully!")
# Config
WINDOW_SIZE = 60  # Number of frames model expects (T)
NUM_CLASSES = 6   # 3 exercises x 2 correctness
CLASSES = ["RTK Correct", "RTK Incorrect", "ELK Correct", "ELK Incorrect", "CTK Correct", "CTK Incorrect"]

# 1. Load Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RehabActionRecognizer(num_classes=NUM_CLASSES).to(device)
model.load_state_dict(torch.load('blazepose_rehab_model_best.pth', map_location=device)) # Load trained weights
model.eval()

# 2. Initialize Virtual Sensor
#sensor = VirtualSensor()
cap = cv2.VideoCapture(0) # 0 for Webcam

buffer = []

print("Starting Rehabilitation Recognition...")

while True:
    ret, frame = cap.read()
    if not ret: break

    # A. Get Skeleton
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    results = detector.detect(mp_image)

    if results.pose_landmarks:
        landmarks = [[lm.x, lm.y, lm.z] for lm in results.pose_landmarks[0]]
        buffer.append(landmarks)
        
        # Maintain sliding window buffer
        if len(buffer) > WINDOW_SIZE:
            buffer.pop(0)

        # B. Inference (only when buffer is full)
        if len(buffer) == WINDOW_SIZE:
            # Prepare tensor: (1, 3, T, 33)
            input_tensor = np.array(buffer) # (T, 33, 3)
            input_tensor = np.transpose(input_tensor, (2, 0, 1)) # (3, T, 33)
            input_tensor = torch.tensor(input_tensor).unsqueeze(0).float().to(device)

            with torch.no_grad():
                output = model(input_tensor)
                prediction = torch.argmax(output, dim=1).item()
                confidence = torch.softmax(output, dim=1)[0][prediction].item()

            # Display Result
            label = f"{CLASSES[prediction]} ({confidence:.2f})"
            cv2.putText(frame, label, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow('Rehab AI', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

#cap.release()
#cv2.destroyAllWindows()

Lite model loaded successfully!
Starting Rehabilitation Recognition...


In [ ]:
import mediapipe as mp
print(hasattr(mp, 'solutions'))
print(dir(mp))

False
['Image', 'ImageFormat', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'tasks']


: 